# FX40 spectroscopy loader

Loads TIF files from the NX40 using their provided code and converts them to a dataframe

Make sure TIFF_Read_Write.py is in the same folder

TIFF_Read_Write.py found [here](https://spmdevhub.com/snippets/reading-and-writing-tiff-files/) on 23-09-2026

In [2]:
import os
import numpy as np
from pspylib.tiff.reader import TiffReader
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from TIFF_Read_Write import Read_tiff

g_0 = 7.748e-5

## Load with no processing

In [5]:
dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 3hr\AFM\Memristor measurements\calibration\0V - 2V"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]
    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 3hr\AFM\Memristor measurements\calibration\0V - 2V/


  0%|          | 0/500 [00:00<?, ?it/s]

C:\Users\newsonj\AppData\Local\Temp\ipykernel_32124\2173294173.py:22: RuntimeWarning: divide by zero encountered in log10
  logg = np.log10(np.abs(g/g_0))


               filename                                               Bias  \
0  iv_261014_1501_.tiff  [0.0009599514305591583, 0.003999900072813034, ...   
1  iv_261014_1502_.tiff  [0.0009599514305591583, 0.003999900072813034, ...   
2  iv_261014_1503_.tiff  [0.0009599514305591583, 0.003999900072813034, ...   
3  iv_261014_1504_.tiff  [0.0009599514305591583, 0.003999900072813034, ...   
4  iv_261014_1505_.tiff  [0.0009599514305591583, 0.003999900072813034, ...   

                                             Current  \
0  [0.016406282782554626, 0.02775910496711731, 0....   
1  [-0.024804741144180298, 0.002734466688707471, ...   
2  [0.0664648711681366, 0.023391202092170715, 0.0...   
3  [-0.00846934411674738, 0.011311947368085384, 0...   
4  [0.017421424388885498, 0.006497010122984648, 0...   

                                                dIdV  \
0  [3.734544073134567e-09, 5.959017159919522e-10,...   
1  [9.059102989473427e-09, 5.717485117657635e-09,...   
2  [-1.4169209465338287e-0

## load and filter data

In [9]:
dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\-2V - 2V\Area 3"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

SCBiasRange = [0,1] # define the bias range for filtering short circuits
OCBiasRange = [1.8,2] # define the bias range for filtering open circuits

maxCurrent = 9900 # max current in nA for filtering short circuits
minCurrent = 0.5 # min current in nA for filtering open circuits

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]

    if any((SCBiasRange[0] < np.abs(v[i]) < SCBiasRange[1] and np.abs(j[i]) > maxCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the short circuit condition

    if any((OCBiasRange[0] < np.abs(v[i]) < OCBiasRange[1] and np.abs(j[i]) < minCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the open circuit condition

    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data_Filtered.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\-2V - 2V\Area 3/


  0%|          | 0/109 [00:00<?, ?it/s]

C:\Users\newsonj\AppData\Local\Temp\ipykernel_41244\2831184453.py:35: RuntimeWarning: divide by zero encountered in log10
  logg = np.log10(np.abs(g/g_0))


                     filename  \
0  -2V - 2V_261002_705__.tiff   
1  -2V - 2V_261002_706__.tiff   
2  -2V - 2V_261002_707__.tiff   
3  -2V - 2V_261002_708__.tiff   
4  -2V - 2V_261002_709__.tiff   

                                                Bias  \
0  [0.0019198935478925705, 0.007839822210371494, ...   
1  [0.0019198935478925705, 0.007839822210371494, ...   
2  [0.0019198935478925705, 0.007839822210371494, ...   
3  [0.0019198935478925705, 0.007839822210371494, ...   
4  [0.0019198935478925705, 0.007839822210371494, ...   

                                             Current  \
0  [2.771103858947754, 2.761920928955078, 2.76318...   
1  [2.7465169429779053, 2.778144598007202, 2.7459...   
2  [2.771615982055664, 2.7711503505706787, 2.7461...   
3  [2.771597385406494, 2.7550292015075684, 2.7579...   
4  [2.7327332496643066, 2.7518906593322754, 2.794...   

                                                dIdV  \
0  [-1.5511892991005473e-09, -8.143211448762205e-...   
1  [5.342573674

## Load and subtract a reference average

In [8]:
calibrationFile = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Testing\2026-09-23 Memristor testing\Force dependent IV\Calibration\Processed\FX40_Spectroscopy_Data.pkl"
calib = pd.read_pickle(calibrationFile)
currentAvg = np.nanmean(np.array(calib["Current"].tolist()),axis=0)

dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Testing\2026-09-23 Memristor testing\Force dependent IV\0.5V Area 2"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]-currentAvg
    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data_corrected.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Testing\2026-09-23 Memristor testing\Force dependent IV\0.5V Area 2/


  0%|          | 0/501 [00:00<?, ?it/s]

                         filename  \
0  fP2 IV test_260923_3419__.tiff   
1  fP2 IV test_260923_3420__.tiff   
2  fP2 IV test_260923_3421__.tiff   
3  fP2 IV test_260923_3422__.tiff   
4  fP2 IV test_260923_3423__.tiff   

                                                Bias  \
0  [0.0019198935478925705, 0.007839822210371494, ...   
1  [0.0019198935478925705, 0.007839822210371494, ...   
2  [0.0019198935478925705, 0.007839822210371494, ...   
3  [0.0019198935478925705, 0.007839822210371494, ...   
4  [0.0019198935478925705, 0.007839822210371494, ...   

                                             Current  \
0  [-0.005212991964071989, -0.00556654962484673, ...   
1  [-0.03524814499542117, -0.003973988022254262, ...   
2  [0.00029112445190548897, 0.007322954807831492,...   
3  [-0.031047880183905363, 0.002657028709007945, ...   
4  [0.01807938562706113, 0.007155316744400706, 0....   

                                                dIdV  \
0  [-5.972329751465152e-11, 2.069723256581785e-

## Filter and calibrate files

In [ ]:
calibrationFile = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Testing\2026-09-23 Memristor testing\Force dependent IV\Calibration\Processed\FX40_Spectroscopy_Data.pkl"
calib = pd.read_pickle(calibrationFile)
currentAvg = np.nanmean(np.array(calib["Current"].tolist()),axis=0)

dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\-2V - 2V\Area 1"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

SCBiasRange = [0,1] # define the bias range for filtering short circuits
OCBiasRange = [1.8,2] # define the bias range for filtering open circuits

maxCurrent = 9900 # max current in nA for filtering short circuits
minCurrent = 0.5 # min current in nA for filtering open circuits

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]

    if any((SCBiasRange[0] < np.abs(v[i]) < SCBiasRange[1] and np.abs(j[i]) > maxCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the short circuit condition

    if any((OCBiasRange[0] < np.abs(v[i]) < OCBiasRange[1] and np.abs(j[i]) < minCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the open circuit condition

    j = j-currentAvg
    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data_Filtered_Calibrated.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\-2V - 2V\Area 1/


  0%|          | 0/111 [00:00<?, ?it/s]

                     filename  \
0  -2V - 2V_261002_004__.tiff   
1  -2V - 2V_261002_005__.tiff   
2  -2V - 2V_261002_006__.tiff   
3  -2V - 2V_261002_007__.tiff   
4  -2V - 2V_261002_008__.tiff   

                                                Bias  \
0  [0.0019198935478925705, 0.007839822210371494, ...   
1  [0.0019198935478925705, 0.007839822210371494, ...   
2  [0.0019198935478925705, 0.007839822210371494, ...   
3  [0.0019198935478925705, 0.007839822210371494, ...   
4  [0.0019198935478925705, 0.007839822210371494, ...   

                                             Current  \
0  [2.7352643427439034, 2.7419421960156694, 2.725...   
1  [2.7730668005533516, 2.7307851601880326, 2.731...   
2  [2.706114095170051, 2.708470611695601, 2.74658...   
3  [2.7172711309976876, 2.737835197572066, 2.7116...   
4  [2.7315669474191964, 2.7322006512921586, 2.766...   

                                                dIdV  \
0  [1.1280293484093686e-09, -2.334849043830152e-1...   
1  [-7.14225504

# Memristor script loader

In [4]:
baseDir = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-12 new 100ugml 3hr\AFM 2\Memristor measurements"

sequence = ["WIDE", "WRITE", "READW", "ERASE", "READE", "WIDE"]

calibrationFiles = [r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-12 new 100ugml 3hr\AFM\calibration\-1.5V - 2V\Processed\FX40_Spectroscopy_Data.pkl",#wide
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-12 new 100ugml 3hr\AFM\calibration\-1.5V - 0V\Processed\FX40_Spectroscopy_Data.pkl",#write
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-12 new 100ugml 3hr\AFM\calibration\-0.5V - 0V\Processed\FX40_Spectroscopy_Data.pkl",#read
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-12 new 100ugml 3hr\AFM\calibration\-0.5V - 0V\Processed\FX40_Spectroscopy_Data.pkl",#read
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-12 new 100ugml 3hr\AFM\calibration\0V - 2V\Processed\FX40_Spectroscopy_Data.pkl"]#erase

reader = Read_tiff()

calib = [pd.read_pickle(f) for f in calibrationFiles]
currentAvgs = {"WIDE": np.nanmean(np.array(calib[0]["Current"].tolist()),axis=0),
               "WRITE": np.nanmean(np.array(calib[1]["Current"].tolist()),axis=0),
               "READW": np.nanmean(np.array(calib[2]["Current"].tolist()),axis=0),
               "READE": np.nanmean(np.array(calib[3]["Current"].tolist()),axis=0),
               "ERASE": np.nanmean(np.array(calib[4]["Current"].tolist()),axis=0)}

folders = sorted(
    f for f in os.listdir(baseDir)
    if os.path.isdir(os.path.join(baseDir, f))
)

for folder in tqdm(folders):
    files = [f for f in os.listdir(os.path.join(baseDir, folder)) if f.endswith('.tiff')]
    files.sort(key=lambda f: int(os.path.splitext(f)[0].rstrip("_").rsplit("_", 1)[-1]))
    #print(files)

    filesSplit = [files[i:i+len(sequence)] for i in range(0, len(files), len(sequence))]
    print(folder + ": " + str(len(filesSplit)) + " loops")
    #print(filesSplit)

    data = []

    for group in filesSplit:
        valid = True
        groupData = []
        for i, file in enumerate(group):
            spectDict, spectTiff = reader(os.path.join(baseDir, folder, file))
            v = spectDict['SPECT_DATA'][0][0]
            j = spectDict['SPECT_DATA'][0][1]-currentAvgs[sequence[i]]
            g = np.gradient(j*1e-9,v)
            logg = np.log10(np.abs(g/g_0))
            logj = np.log10(np.abs(j*1e-9))
            if np.max(np.abs(j))>9900:
                valid = False
                print(f"Invalid data in {file}, skipping loop {group}")
                break
            groupData.append({
                "filename": file,
                "Type": sequence[i],
                "Bias": v.tolist(),
                "Current": (j).tolist(), 
                "dIdV": g.tolist(),
                "Log(G/G0)": logg.tolist(),
                "Log(Current)": logj.tolist()
            })
        if valid:
            data.extend(groupData)

    if not os.path.exists(os.path.join(baseDir, folder, 'Processed')):
        os.makedirs(os.path.join(baseDir, folder, 'Processed'))
    df_all = pd.DataFrame(data)
    df_all.to_pickle(os.path.join(baseDir, folder, 'Processed', 'FX40_Spectroscopy_Data_corrected.pkl'))




  0%|          | 0/22 [00:00<?, ?it/s]

0: 200 loops
1: 200 loops
10: 200 loops
11: 200 loops
12: 167 loops
Invalid data in iv_ 261016_15398_.tiff, skipping loop ['iv_ 261016_15398_.tiff', 'iv_ 261016_15399_.tiff', 'iv_ 261016_15400_.tiff', 'iv_ 261016_15401_.tiff', 'iv_ 261016_15402_.tiff', 'iv_ 261016_15403_.tiff']
13: 200 loops
14: 200 loops
15: 64 loops
Invalid data in iv_ 261016_17807_.tiff, skipping loop ['iv_ 261016_17804_.tiff', 'iv_ 261016_17805_.tiff', 'iv_ 261016_17806_.tiff', 'iv_ 261016_17807_.tiff', 'iv_ 261016_17808_.tiff', 'iv_ 261016_17809_.tiff']
Invalid data in iv_ 261016_17810_.tiff, skipping loop ['iv_ 261016_17810_.tiff', 'iv_ 261016_17811_.tiff', 'iv_ 261016_17812_.tiff', 'iv_ 261016_17813_.tiff', 'iv_ 261016_17814_.tiff', 'iv_ 261016_17815_.tiff']
Invalid data in iv_ 261016_17816_.tiff, skipping loop ['iv_ 261016_17816_.tiff', 'iv_ 261016_17817_.tiff', 'iv_ 261016_17818_.tiff', 'iv_ 261016_17819_.tiff', 'iv_ 261016_17820_.tiff', 'iv_ 261016_17821_.tiff']
Invalid data in iv_ 261016_17822_.tiff, skippin